# 04 — Demand Forecasting: Baseline vs Enriched
**Demand Signal Feature Store — Capstone Project**

This notebook trains two forecasting models and compares their MAPE:
1. **Baseline model** — uses only ERP tabular features
2. **Enriched model** — uses ERP features + LLM-extracted signals

Target variable: `avg_delay` (average delay days per supplier per month).
The enriched model should show lower MAPE, proving that qualitative supplier signals improve forecasts.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error
from xgboost import XGBRegressor

sns.set_style("whitegrid")

## 1. Load Feature Store

In [ ]:
feature_store = pd.read_csv("../data/feature_store.csv")
print(f"Feature store: {feature_store.shape}")
feature_store.head()

## 2. Prepare Features

In [ ]:
TARGET = "avg_delay"

# ERP-only features (baseline)
ERP_FEATURES = [
    "total_orders",
    "avg_lead_time",
    "max_delay",
    "on_time_rate",
    "total_qty_ordered",
    "total_qty_received",
    "avg_unit_cost",
    "fulfillment_rate",
]

# LLM-extracted features (enrichment)
LLM_FEATURES = [
    "risk_mention_rate",
    "avg_delay_signal",
    "max_delay_signal",
    "avg_sentiment",
    "min_sentiment",
    "capacity_is_constrained",
    "note_count",
]

ENRICHED_FEATURES = ERP_FEATURES + LLM_FEATURES

print(f"Baseline features ({len(ERP_FEATURES)}): {ERP_FEATURES}")
print(f"Enriched features ({len(ENRICHED_FEATURES)}): {ENRICHED_FEATURES}")

In [ ]:
# Drop rows where target or features are missing
df = feature_store.dropna(subset=[TARGET] + ENRICHED_FEATURES).copy()

# Sort by time for temporal split
df = df.sort_values("year_month").reset_index(drop=True)
print(f"Rows after dropping NaN: {len(df)}")
print(f"Time range: {df['year_month'].min()} to {df['year_month'].max()}")

## 3. Train-Test Split (Temporal)

In [ ]:
# Use last 3 months as test set (temporal split, not random)
all_months = sorted(df["year_month"].unique())
test_months = all_months[-3:]
train_months = all_months[:-3]

train_df = df[df["year_month"].isin(train_months)]
test_df = df[df["year_month"].isin(test_months)]

print(f"Train: {len(train_df)} rows ({train_months[0]} to {train_months[-1]})")
print(f"Test:  {len(test_df)} rows ({test_months[0]} to {test_months[-1]})")

## 4. Train Baseline Model (ERP Only)

In [ ]:
X_train_base = train_df[ERP_FEATURES]
X_test_base = test_df[ERP_FEATURES]
y_train = train_df[TARGET]
y_test = test_df[TARGET]

baseline_model = XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
)
baseline_model.fit(X_train_base, y_train)

y_pred_base = baseline_model.predict(X_test_base)

# Avoid division by zero in MAPE — replace 0 targets with small value
y_test_safe = y_test.replace(0, 0.1)

mape_base = mean_absolute_percentage_error(y_test_safe, y_pred_base)
mae_base = mean_absolute_error(y_test, y_pred_base)

print(f"=== Baseline Model (ERP Only) ===")
print(f"  MAPE: {mape_base:.2%}")
print(f"  MAE:  {mae_base:.2f} days")

## 5. Train Enriched Model (ERP + LLM Features)

In [ ]:
X_train_enriched = train_df[ENRICHED_FEATURES]
X_test_enriched = test_df[ENRICHED_FEATURES]

enriched_model = XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
)
enriched_model.fit(X_train_enriched, y_train)

y_pred_enriched = enriched_model.predict(X_test_enriched)

mape_enriched = mean_absolute_percentage_error(y_test_safe, y_pred_enriched)
mae_enriched = mean_absolute_error(y_test, y_pred_enriched)

print(f"=== Enriched Model (ERP + LLM) ===")
print(f"  MAPE: {mape_enriched:.2%}")
print(f"  MAE:  {mae_enriched:.2f} days")

## 6. MAPE Comparison

In [ ]:
improvement = (mape_base - mape_enriched) / mape_base * 100

print("=" * 50)
print("FORECAST PERFORMANCE COMPARISON")
print("=" * 50)
print(f"{'Model':<25} {'MAPE':>10} {'MAE':>10}")
print("-" * 50)
print(f"{'Baseline (ERP only)':<25} {mape_base:>9.2%} {mae_base:>9.2f}")
print(f"{'Enriched (ERP + LLM)':<25} {mape_enriched:>9.2%} {mae_enriched:>9.2f}")
print("-" * 50)
print(f"{'MAPE improvement':<25} {improvement:>9.1f}%")
print("=" * 50)

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# MAPE comparison bar chart
models = ["Baseline\n(ERP only)", "Enriched\n(ERP + LLM)"]
mapes = [mape_base, mape_enriched]
colors = ["#d4726a", "#5b9f5b"]
axes[0].bar(models, mapes, color=colors, width=0.5)
axes[0].set_ylabel("MAPE")
axes[0].set_title("MAPE Comparison")
for i, v in enumerate(mapes):
    axes[0].text(i, v + 0.01, f"{v:.2%}", ha="center", fontweight="bold")

# Actual vs predicted — Baseline
axes[1].scatter(y_test, y_pred_base, alpha=0.6, color=colors[0], label="Baseline")
axes[1].plot([0, y_test.max()], [0, y_test.max()], "k--", alpha=0.5)
axes[1].set_xlabel("Actual Delay (days)")
axes[1].set_ylabel("Predicted Delay (days)")
axes[1].set_title("Baseline: Actual vs Predicted")

# Actual vs predicted — Enriched
axes[2].scatter(y_test, y_pred_enriched, alpha=0.6, color=colors[1], label="Enriched")
axes[2].plot([0, y_test.max()], [0, y_test.max()], "k--", alpha=0.5)
axes[2].set_xlabel("Actual Delay (days)")
axes[2].set_ylabel("Predicted Delay (days)")
axes[2].set_title("Enriched: Actual vs Predicted")

plt.tight_layout()
plt.savefig("../data/mape_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to ../data/mape_comparison.png")

## 7. Feature Importance — What LLM Signals Matter Most?

In [ ]:
importance = pd.DataFrame({
    "feature": ENRICHED_FEATURES,
    "importance": enriched_model.feature_importances_,
}).sort_values("importance", ascending=True)

# Color LLM features differently
importance["is_llm"] = importance["feature"].isin(LLM_FEATURES)

fig, ax = plt.subplots(figsize=(10, 6))
bar_colors = ["#5b9f5b" if llm else "#6a8fc4" for llm in importance["is_llm"]]
ax.barh(importance["feature"], importance["importance"], color=bar_colors)
ax.set_xlabel("Feature Importance")
ax.set_title("Feature Importance — Enriched Model\n(Green = LLM-extracted, Blue = ERP)")
plt.tight_layout()
plt.savefig("../data/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Cross-Validation (Optional — More Robust MAPE)

In [ ]:
tscv = TimeSeriesSplit(n_splits=4)

cv_results = {"baseline": [], "enriched": []}

for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
    X_tr_b = df.iloc[train_idx][ERP_FEATURES]
    X_te_b = df.iloc[test_idx][ERP_FEATURES]
    X_tr_e = df.iloc[train_idx][ENRICHED_FEATURES]
    X_te_e = df.iloc[test_idx][ENRICHED_FEATURES]
    y_tr = df.iloc[train_idx][TARGET]
    y_te = df.iloc[test_idx][TARGET]
    y_te_safe = y_te.replace(0, 0.1)

    m_base = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
    m_base.fit(X_tr_b, y_tr)
    cv_results["baseline"].append(mean_absolute_percentage_error(y_te_safe, m_base.predict(X_te_b)))

    m_enr = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
    m_enr.fit(X_tr_e, y_tr)
    cv_results["enriched"].append(mean_absolute_percentage_error(y_te_safe, m_enr.predict(X_te_e)))

print(f"Cross-validated MAPE (4-fold TimeSeriesSplit):")
print(f"  Baseline: {np.mean(cv_results['baseline']):.2%} (+/- {np.std(cv_results['baseline']):.2%})")
print(f"  Enriched: {np.mean(cv_results['enriched']):.2%} (+/- {np.std(cv_results['enriched']):.2%})")